# SATP Perpetrator Classification - Revised

This notebook implements multi-class text classification for identifying perpetrators in SATP incidents.
The structure follows a clean, organized approach:

1. **Import Libraries** - All necessary dependencies
2. **Define Functions & Classes** - Reusable components
3. **Import Data** - Load and preview data
4. **Clean Data** - Data preprocessing
5. **Train-Test Split** - Create datasets
6. **Model Training** - Run experiments
7. **Save Results** - Export outcomes
8. **Analysis & Visualization** - Summary and plots


## 1. Import Libraries


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import json
import os
import shutil
from pathlib import Path

# Machine Learning
import torch
from torch.utils.data import Dataset
from torch.nn.functional import softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available. Using: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")


## 2. Define Functions & Classes


In [ ]:
class SingleLabelDataset(Dataset):
    """
    Custom Dataset class for single-label multiclass classification.
    
    Args:
        texts: List of text strings
        labels: List of integer class labels (e.g., [0, 1, 2, ...])
        tokenizer: A Hugging Face AutoTokenizer
        max_length: Max sequence length for tokenization
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Squeeze batch dimension
        item = {k: v.squeeze() for k, v in encoding.items()}
        # Integer labels for cross-entropy
        item["labels"] = torch.tensor(label, dtype=torch.long)

        return item


In [ ]:
def compute_metrics(eval_pred, label_names=None):
    """
    Compute comprehensive classification metrics.
    
    Returns:
        dict: Contains accuracy, f1_macro, f1_weighted, and full classification report
    """
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=1).numpy()

    # Accuracy
    acc = accuracy_score(labels, preds)

    # Full classification report (dict)
    report_dict = classification_report(
        labels,
        preds,
        target_names=label_names if label_names else None,
        zero_division=0,
        output_dict=True
    )

    # Print readable version
    if label_names:
        print("\nFull Classification Report:\n",
              classification_report(labels, preds, target_names=label_names, zero_division=0))
    else:
        print("\nFull Classification Report:\n",
              classification_report(labels, preds, zero_division=0))

    return {
        "accuracy": acc,
        "f1_macro": report_dict["macro avg"]["f1-score"],
        "f1_weighted": report_dict["weighted avg"]["f1-score"],
        "class_report": json.dumps(report_dict)
    }


In [ ]:
def train_multiclass_model(
    train_df,
    val_df,
    test_df,
    text_col="incident_summary",
    label_col="perpetrator",
    model_name="bert-base-uncased",
    epochs=2,
    batch_size=8,
    max_length=128
):
    """
    Train a multi-class classifier with separate train, val, and test sets.
    
    Args:
        train_df, val_df, test_df: DataFrames with text and labels
        text_col: Name of the column containing the text
        label_col: Name of the column containing the class label
        model_name: HF model identifier
        epochs: Number of training epochs
        batch_size: Batch size for training and evaluation
        max_length: Maximum sequence length for tokenization
    
    Returns:
        tuple: (model, tokenizer, test_metrics, predictions_df, id2label)
    """
    print(f"\nTraining {model_name} for {epochs} epochs...")
    
    # Convert labels to integer IDs
    unique_labels = train_df[label_col].unique()
    label2id = {label: i for i, label in enumerate(unique_labels)}
    id2label = {v: k for k, v in label2id.items()}

    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()
    
    train_df["label_id"] = train_df[label_col].map(label2id)
    val_df["label_id"] = val_df[label_col].map(label2id)
    test_df["label_id"] = test_df[label_col].map(label2id)

    # Prepare data lists
    train_texts = train_df[text_col].tolist()
    train_labels = train_df["label_id"].tolist()
    val_texts = val_df[text_col].tolist()
    val_labels = val_df["label_id"].tolist()
    test_texts = test_df[text_col].tolist()
    test_labels = test_df["label_id"].tolist()

    # Initialize tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    num_labels = len(unique_labels)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    model.to(device)

    # Create Dataset objects
    train_dataset = SingleLabelDataset(train_texts, train_labels, tokenizer, max_length)
    val_dataset = SingleLabelDataset(val_texts, val_labels, tokenizer, max_length)
    test_dataset = SingleLabelDataset(test_texts, test_labels, tokenizer, max_length)

    # Training arguments
    training_args = TrainingArguments(
        output_dir="./model_output",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        logging_steps=10,
        report_to="none",
        load_best_model_at_end=False
    )

    # Create trainer
    label_names = [id2label[i] for i in range(num_labels)]

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=lambda eval_pred: compute_metrics(eval_pred, label_names=label_names)
    )

    # Train model
    trainer.train()

    # Evaluate on test set
    test_metrics = trainer.evaluate(test_dataset)
    print(f"\nTest Set Evaluation Results: {test_metrics}")

    # Generate predictions
    predictions_output = trainer.predict(test_dataset)
    logits = predictions_output.predictions
    probs = softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)

    # Create predictions DataFrame
    pred_df = pd.DataFrame({
        "incident_summary": test_df[text_col].tolist(),
        "true_label_id": test_labels,
        "true_label": [id2label[i] for i in test_labels],
        "pred_label_id": preds.tolist(),
        "pred_label": [id2label[i] for i in preds],
        "logits": logits.tolist(),
        "probabilities": probs.tolist()
    })

    return trainer.model, tokenizer, test_metrics, pred_df, id2label


In [ ]:
def run_experiments(
    train_df, 
    val_df, 
    test_df, 
    fractions=[1/32, 1/16, 1/8, 1/4, 1/2, 1.0],
    models_list=[
        "bert-base-cased",
        "snowood1/ConfliBERT-scr-cased",
        "FacebookAI/roberta-base",
        "distilbert-base-cased",
        "xlnet-base-cased",
        "google/electra-base-discriminator"
    ],
    epochs=2,
    batch_size=32
):
    """
    Run experiments across different data fractions and models.
    
    Returns:
        tuple: (results_df, predictions_df)
    """
    
    # Define labels for fractions and models
    fraction_labels = {
        1/32: "3%", 1/16: "6%", 1/8: "12%", 
        1/4: "25%", 1/2: "50%", 1.0: "100%"
    }
    
    model_name_labels = {
        "bert-base-cased": "BERT",
        "snowood1/ConfliBERT-scr-cased": "ConfliBERT",
        "FacebookAI/roberta-base": "RoBERTa",
        "distilbert-base-cased": "DistilBERT",
        "xlnet-base-cased": "XLNet",
        "google/electra-base-discriminator": "ELECTRA"
    }
    
    results_list = []
    all_predictions = []

    for frac in fractions:
        subset_size = int(len(train_df) * frac)
        train_subset = train_df.sample(n=subset_size, random_state=42)
        
        frac_label = fraction_labels.get(frac, f"{frac*100:.1f}%")
        print(f"\n=== DATA FRACTION: {frac} ({subset_size} rows) ===")

        for model_name in models_list:
            model_label = model_name_labels.get(model_name, model_name)
            print(f"\nTraining model: {model_label}")

            try:
                model, tokenizer, test_metrics, pred_df, id2label = train_multiclass_model(
                    train_subset,
                    val_df,
                    test_df,
                    text_col="incident_summary",
                    label_col="perpetrator",
                    model_name=model_name,
                    epochs=epochs,
                    batch_size=batch_size
                )

                # Store results
                run_result = {
                    "fraction_raw": frac,
                    "fraction_label": frac_label,
                    "subset_size": subset_size,
                    "model_raw": model_name,
                    "model_label": model_label
                }

                # Add metrics
                for key, val in test_metrics.items():
                    if key != "class_report":  # Skip the JSON string for now
                        run_result[key] = val

                # Add training support counts
                label_counts = train_subset["perpetrator"].value_counts()
                for label_name, count in label_counts.items():
                    run_result[f"train_support_{label_name}"] = count

                results_list.append(run_result)

                # Add experiment info to predictions
                pred_df["model"] = model_name
                pred_df["model_label"] = model_label
                pred_df["fraction"] = frac
                pred_df["fraction_label"] = frac_label
                all_predictions.append(pred_df)
                
            except Exception as e:
                print(f"Error training {model_name} with fraction {frac}: {e}")
                continue

    results_df = pd.DataFrame(results_list)
    predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
    
    return results_df, predictions_df


In [ ]:
# Load data from GitHub (primary source)
url = 'https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/perpetrator.csv'

try:
    df = pd.read_csv(url)
    print("✓ Successfully loaded data from GitHub")
    print(f"Dataset shape: {df.shape}")
    print("\nFirst few rows:")
    print(df.head())
except Exception as e:
    print(f"❌ Error loading CSV from URL: {e}")
    
    # Fallback to local file if available
    local_path = "../../data/perpetrator.csv"
    if Path(local_path).exists():
        df = pd.read_csv(local_path)
        print(f"✓ Loaded from local file: {local_path}")
    else:
        raise FileNotFoundError("Could not load data from URL or local file")


In [ ]:
# Explore the data
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nUnique perpetrator values: {df['perpetrator'].unique()}")
print(f"\nPerpetrator distribution:")
print(df['perpetrator'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())


In [ ]:
# Data cleaning
print("Data cleaning steps:")

# 1. Select necessary columns
df_clean = df[['incident_summary', 'perpetrator']].copy()
print(f"1. Selected columns: {list(df_clean.columns)}")

# 2. Remove missing values
initial_shape = df_clean.shape[0]
df_clean.dropna(inplace=True)
final_shape = df_clean.shape[0]
print(f"2. Removed {initial_shape - final_shape} rows with missing values")

# 3. Remove potential duplicates
initial_shape = df_clean.shape[0]
df_clean.drop_duplicates(inplace=True)
final_shape = df_clean.shape[0]
print(f"3. Removed {initial_shape - final_shape} duplicate rows")

# 4. Clean text (basic preprocessing)
df_clean['incident_summary'] = df_clean['incident_summary'].str.strip()
print("4. Cleaned whitespace from text")

print(f"\nFinal cleaned dataset shape: {df_clean.shape}")
print(f"Final perpetrator distribution:")
print(df_clean['perpetrator'].value_counts())


In [ ]:
# Create stratified train-validation-test splits
print("Creating train-validation-test splits...")

# First split: separate training data from temp (test + validation)
train_df, temp_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    stratify=df_clean['perpetrator'], 
    random_state=42
)

# Second split: divide temp into validation and test
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['perpetrator'], 
    random_state=42
)

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Verify stratification
print("\n=== Label Distribution Verification ===")
print("\nTraining set distribution:")
train_dist = train_df['perpetrator'].value_counts(normalize=True).sort_index()
print(train_dist)

print("\nValidation set distribution:")
val_dist = val_df['perpetrator'].value_counts(normalize=True).sort_index()
print(val_dist)

print("\nTest set distribution:")
test_dist = test_df['perpetrator'].value_counts(normalize=True).sort_index()
print(test_dist)

# Show absolute counts
print("\n=== Absolute Counts ===")
print("Training:", train_df['perpetrator'].value_counts().sort_index())
print("Validation:", val_df['perpetrator'].value_counts().sort_index())
print("Test:", test_df['perpetrator'].value_counts().sort_index())


In [ ]:
# Configuration for experiments
EXPERIMENT_CONFIG = {
    'fractions': [1/32, 1/16, 1/8, 1/4, 1/2, 1.0],  # Data fractions to test
    'models': [
        "bert-base-cased",
        "snowood1/ConfliBERT-scr-cased", 
        "FacebookAI/roberta-base",
        "distilbert-base-cased",
        "xlnet-base-cased",
        "google/electra-base-discriminator"
    ],
    'epochs': 2,
    'batch_size': 32
}

print("Experiment Configuration:")
print(f"Data fractions: {EXPERIMENT_CONFIG['fractions']}")
print(f"Models: {len(EXPERIMENT_CONFIG['models'])} models")
print(f"Epochs: {EXPERIMENT_CONFIG['epochs']}")
print(f"Batch size: {EXPERIMENT_CONFIG['batch_size']}")
print(f"\nTotal experiments: {len(EXPERIMENT_CONFIG['fractions']) * len(EXPERIMENT_CONFIG['models'])}")


In [ ]:
# Run all experiments
print("🚀 Starting experiment loop...\n")

results_df, predictions_df = run_experiments(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    fractions=EXPERIMENT_CONFIG['fractions'],
    models_list=EXPERIMENT_CONFIG['models'],
    epochs=EXPERIMENT_CONFIG['epochs'],
    batch_size=EXPERIMENT_CONFIG['batch_size']
)

print("\n✅ All experiments completed!")
print(f"Results shape: {results_df.shape}")
print(f"Predictions shape: {predictions_df.shape}")


In [ ]:
# Create output directory
output_dir = Path("../../results/perpetrator")
output_dir.mkdir(parents=True, exist_ok=True)

# Save results
results_path = output_dir / "perpetrator_summary.csv"
predictions_path = output_dir / "perpetrator_predictions.csv"

results_df.to_csv(results_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

print(f"✅ Results saved to:")
print(f"  Summary: {results_path}")
print(f"  Predictions: {predictions_path}")

# Also save as JSON for easier programmatic access
results_json_path = output_dir / "perpetrator_summary.json"
results_df.to_json(results_json_path, orient="records", indent=2)
print(f"  JSON: {results_json_path}")


In [ ]:
# Define visualization functions
def plot_performance_by_fraction(results_df, metric='eval_accuracy'):
    """Plot model performance vs data fraction."""
    plt.figure(figsize=(12, 8))
    sns.lineplot(
        data=results_df, 
        x="fraction_label", 
        y=metric, 
        hue="model_label", 
        marker="o",
        linewidth=2,
        markersize=8
    )
    
    plt.title(f"{metric.replace('eval_', '').replace('_', ' ').title()} vs. Data Fraction", 
              fontsize=16, fontweight='bold')
    plt.xlabel("Data Fraction", fontsize=14)
    plt.ylabel(metric.replace('eval_', '').replace('_', ' ').title(), fontsize=14)
    plt.xticks(rotation=45)
    plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_accuracy_vs_speed(results_df):
    """Create scatter plot of accuracy vs. throughput."""
    plt.figure(figsize=(12, 8))
    
    scatter = sns.scatterplot(
        data=results_df,
        x="eval_samples_per_second",
        y="eval_accuracy",
        hue="model_label",
        size="fraction_raw",
        sizes=(50, 400),
        alpha=0.7
    )
    
    plt.title("Model Performance: Accuracy vs. Throughput", 
              fontsize=16, fontweight='bold')
    plt.xlabel("Throughput (samples/second)", fontsize=14)
    plt.ylabel("Accuracy", fontsize=14)
    
    # Add trend line
    sns.regplot(data=results_df, x="eval_samples_per_second", y="eval_accuracy", 
                scatter=False, color='gray', alpha=0.5)
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_heatmap(results_df, metric='eval_accuracy'):
    """Create heatmap of performance across models and data fractions."""
    # Pivot data for heatmap
    heatmap_data = results_df.pivot(
        index="model_label",
        columns="fraction_label",
        values=metric
    )
    
    # Ensure proper column order
    column_order = ["3%", "6%", "12%", "25%", "50%", "100%"]
    heatmap_data = heatmap_data.reindex(columns=column_order)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        cbar_kws={"label": metric.replace('eval_', '').replace('_', ' ').title()},
        linewidths=0.5
    )
    
    plt.title(f"{metric.replace('eval_', '').replace('_', ' ').title()} Heatmap", 
              fontsize=16, fontweight='bold')
    plt.xlabel("Data Fraction", fontsize=14)
    plt.ylabel("Model", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
# Load results for visualization (in case running visualization separately)
try:
    if 'results_df' not in locals() or results_df.empty:
        results_path = Path("../../results/perpetrator/perpetrator_summary.csv")
        results_df = pd.read_csv(results_path)
        print(f"Loaded results from {results_path}")
    
    print(f"Results DataFrame shape: {results_df.shape}")
    print(f"Available columns: {list(results_df.columns)}")
    print(f"\nSample of results:")
    print(results_df[['model_label', 'fraction_label', 'eval_accuracy', 'eval_f1_macro']].head(10))
    
except FileNotFoundError:
    print("⚠️ Results file not found. Please run the experiments first.")


In [ ]:
# Performance by data fraction
print("📊 Generating performance visualizations...\n")

# 1. Accuracy vs Data Fraction
plot_performance_by_fraction(results_df, 'eval_accuracy')


In [ ]:
# 2. F1 Macro vs Data Fraction
plot_performance_by_fraction(results_df, 'eval_f1_macro')


In [ ]:
# 3. F1 Weighted vs Data Fraction
plot_performance_by_fraction(results_df, 'eval_f1_weighted')


In [ ]:
# 4. Accuracy vs Speed Trade-off
if 'eval_samples_per_second' in results_df.columns:
    plot_accuracy_vs_speed(results_df)
else:
    print("⚠️ Speed metrics not available in results")


In [ ]:
# 5. Heatmaps for different metrics
print("🔥 Generating heatmaps...\n")

# Accuracy heatmap
plot_heatmap(results_df, 'eval_accuracy')


In [ ]:
# F1 Macro heatmap
plot_heatmap(results_df, 'eval_f1_macro')


In [ ]:
# F1 Weighted heatmap
plot_heatmap(results_df, 'eval_f1_weighted')


In [ ]:
# Summary statistics
print("📈 Summary Statistics\n")
print("=" * 50)

# Best performing models by metric
metrics = ['eval_accuracy', 'eval_f1_macro', 'eval_f1_weighted']

for metric in metrics:
    if metric in results_df.columns:
        best_result = results_df.loc[results_df[metric].idxmax()]
        print(f"\n🏆 Best {metric.replace('eval_', '').replace('_', ' ').title()}:")
        print(f"   Model: {best_result['model_label']}")
        print(f"   Data Fraction: {best_result['fraction_label']}")
        print(f"   Score: {best_result[metric]:.4f}")

# Average performance by model (across all fractions)
print("\n📊 Average Performance by Model (across all data fractions):")
model_avg = results_df.groupby('model_label')[metrics].mean().sort_values('eval_accuracy', ascending=False)
print(model_avg.round(4))

# Performance improvement with more data
print("\n📈 Performance Improvement (100% vs 3% data):")
improvement = []
for model in results_df['model_label'].unique():
    model_data = results_df[results_df['model_label'] == model]
    perf_3 = model_data[model_data['fraction_label'] == '3%']['eval_accuracy'].values
    perf_100 = model_data[model_data['fraction_label'] == '100%']['eval_accuracy'].values
    
    if len(perf_3) > 0 and len(perf_100) > 0:
        improvement.append({
            'model': model,
            '3%_accuracy': perf_3[0],
            '100%_accuracy': perf_100[0],
            'improvement': perf_100[0] - perf_3[0],
            'improvement_pct': ((perf_100[0] - perf_3[0]) / perf_3[0]) * 100
        })

improvement_df = pd.DataFrame(improvement).sort_values('improvement', ascending=False)
print(improvement_df.round(4))

print("\n✅ Analysis complete!")
